# Ejecución del Pipeline · Bloque 1 (Audio)

Notebook para ejecutar y verificar cada etapa del pipeline de audio paso a paso.

**Orden de ejecución:**
1. Verificar prerrequisitos
2. Descargar vídeo y transcripción oficial
3. Extraer audio (WAV)
4. Transcripción (ASR)
5. Diarización de hablantes
6. Cruce transcript + diarización → `transcript.csv`
7. Emoción de voz → `voice.csv`
8. Síntesis de voz (TTS) → `briefing.mp3` *(solo si B2 generó `summary.json`)*

## 0. Prerrequisitos

Verificar que todo está instalado y configurado antes de empezar.

In [ ]:
# Verificar instalación de dependencias clave
import importlib

deps = ["torch", "torchaudio", "transformers", "pyannote.audio", "soundfile", "yaml", "dotenv"]
for dep in deps:
    try:
        importlib.import_module(dep.replace(".", "_") if dep == "pyannote.audio" else dep)
        print(f"  ✓ {dep}")
    except ImportError:
        print(f"  ✗ {dep} — ejecutar: pip install -r requirements.txt")

In [ ]:
# Verificar ffmpeg
import shutil
if shutil.which("ffmpeg"):
    print("✓ ffmpeg encontrado")
else:
    print("✗ ffmpeg no encontrado — instálalo antes de continuar")

In [ ]:
# Verificar HF_TOKEN en .env
import os
from src.config import ROOT

env_path = ROOT / ".env"
if env_path.exists():
    print(f"✓ .env encontrado en {env_path}")
else:
    print(f"✗ Falta .env — crear {env_path} con: HF_TOKEN=tu_token_aqui")

token = os.environ.get("HF_TOKEN", "")
if token:
    print(f"✓ HF_TOKEN cargado ({token[:8]}...)")
else:
    print("✗ HF_TOKEN no encontrado en variables de entorno")

In [ ]:
# Configuración común
from src.config import ROOT, RAW_DIR, EVENTS_DIR, load_config, event_dir

cfg = load_config()
fecha = cfg["reference_event"]
ref_dir = event_dir(fecha)
ref_dir.mkdir(parents=True, exist_ok=True)

print(f"Evento de referencia: {fecha}")
print(f"Directorio raw:       {RAW_DIR}")
print(f"Directorio evento:    {ref_dir}")

## 1. Descargar vídeo y transcripción oficial

Descarga el vídeo de YouTube (vía `yt-dlp`) y la transcripción oficial del BCE.

In [ ]:
# Descargar vídeo de la rueda de prensa
from scripts.download_ecb import descargar_video

video_path = RAW_DIR / f"{fecha}.mp4"
if video_path.exists():
    print(f"✓ Vídeo ya existe: {video_path}")
else:
    print(f"Descargando vídeo de la rueda {fecha}...")
    video_path = descargar_video(fecha)

In [ ]:
# Descargar transcripción oficial (ground truth para ASR)
from scripts.download_ecb import descargar_transcripcion

transcript_ref_path = RAW_DIR / f"{fecha}_transcript.txt"
if transcript_ref_path.exists():
    print(f"✓ Transcripción ya existe: {transcript_ref_path}")
else:
    print(f"Descargando transcripción oficial...")
    try:
        transcript_ref_path = descargar_transcripcion(fecha)
    except Exception as e:
        print(f"⚠ No se pudo descargar la transcripción: {e}")
        print("  Puedes descargarla manualmente de: https://www.ecb.europa.eu/press/pressconf/")

## 2. Extraer audio (WAV)

Convierte el vídeo MP4 a WAV mono 16kHz, que es el formato que esperan todos los modelos.

In [ ]:
from src.audio.pipeline import extraer_audio

audio_path = RAW_DIR / f"{fecha}.wav"
if audio_path.exists():
    print(f"✓ Audio ya existe: {audio_path}")
else:
    print("Extrayendo audio del vídeo...")
    audio_path = extraer_audio(video_path)
    print(f"✓ Audio extraído: {audio_path}")

# Verificar duración
import soundfile as sf
info = sf.info(str(audio_path))
print(f"  Duración: {info.duration / 60:.1f} minutos")
print(f"  Sample rate: {info.samplerate} Hz")
print(f"  Canales: {info.channels}")

## 3. Transcripción (ASR)

Modelo: **Whisper large-v3-turbo** — transcribe el audio completo con timestamps por segmento.

In [ ]:
from src.audio.transcription import transcribir

print(f"Transcribiendo {audio_path.name} con Whisper large-v3-turbo...")
print("(esto puede tardar varios minutos dependiendo de la GPU)")
segmentos = transcribir(audio_path)
print(f"\n✓ Segmentos transcritos: {len(segmentos)}")
print("\nPrimeros 5 segmentos:")
for s in segmentos[:5]:
    print(f"  [{s['start']:.1f}s - {s['end']:.1f}s] {s['text'][:80]}")

## 4. Diarización de hablantes

Modelo: **pyannote 3.1** — identifica quién habla en cada momento (presidenta, vicepresidente, periodistas).

In [ ]:
from src.audio.diarization import diarizar

print(f"Diarizando {audio_path.name} con pyannote 3.1...")
diarizacion = diarizar(audio_path)
print(f"\n✓ Turnos detectados: {len(diarizacion)}")
print(f"  Hablantes únicos: {diarizacion['speaker_id'].nunique()}")
print("\nPrimeros turnos:")
print(diarizacion.head(10))

## 5. Cruce ASR + Diarización → `transcript.csv`

Asigna a cada segmento transcrito el hablante correspondiente según la diarización.

In [ ]:
from src.audio.diarization import asignar_hablantes
import pandas as pd

transcript = asignar_hablantes(segmentos, diarizacion)
transcript_path = ref_dir / "transcript.csv"
transcript.to_csv(transcript_path, index=False)

print(f"✓ transcript.csv guardado: {transcript_path}")
print(f"  Filas: {len(transcript)}")
print(f"  Columnas: {list(transcript.columns)}")
print("\nPrimeras filas:")
print(transcript.head(10))

## 6. Emoción de voz → `voice.csv`

Modelo: **wav2vec2-audeering** — extrae arousal (activación) y valence (sentimiento) por ventanas temporales.

In [ ]:
from src.audio.voice_emotion import analizar_emocion

print(f"Analizando emoción de voz con wav2vec2-audeering...")
voice = analizar_emocion(audio_path)
voice_path = ref_dir / "voice.csv"
voice.to_csv(voice_path, index=False)

print(f"\n✓ voice.csv guardado: {voice_path}")
print(f"  Ventanas: {len(voice)}")
print(voice.describe())

## 7. Síntesis de voz (TTS) → `briefing.mp3`

Modelo: **Kokoro-82M** — genera el audio del briefing a partir de `summary.json`.

⚠ Este paso solo funciona si el Bloque 2 ya generó `summary.json` con el campo `briefing_tts`.

In [ ]:
summary_path = ref_dir / "summary.json"

if summary_path.exists():
    from src.audio.tts import generar_briefing
    print("Generando briefing de audio con Kokoro...")
    briefing_path = generar_briefing(fecha)
    print(f"✓ briefing.mp3 guardado: {briefing_path}")
else:
    print("⚠ summary.json no existe todavía (lo genera el Bloque 2)")
    print("  El TTS se ejecutará cuando B2 haya generado el resumen.")
    print(f"  Ruta esperada: {summary_path}")

## 8. Verificación final

Comprobar que todos los archivos de salida de B1 se han generado correctamente.

In [ ]:
archivos_b1 = {
    "transcript.csv": ref_dir / "transcript.csv",
    "voice.csv": ref_dir / "voice.csv",
    "briefing.mp3": ref_dir / "briefing.mp3",
}

print(f"Archivos de salida B1 en {ref_dir}:\n")
for nombre, ruta in archivos_b1.items():
    if ruta.exists():
        size_kb = ruta.stat().st_size / 1024
        print(f"  ✓ {nombre} ({size_kb:.0f} KB)")
    else:
        print(f"  ✗ {nombre} — no generado")

print("\n--- Pipeline B1 completado ---")